# 07 — Goal 3: multidimensional structure and robustness

Visualizes participant-clustered feature structure and preserves pairwise support.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: correlation and support matrices. Interpret only estimable pairs; do not impute or erase sparse/failed metrics to make the structure look cleaner.

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("04_analysis") / "goal3")
correlations = read_table(OUTPUT / "04_analysis" / "descriptive" / "pairwise_clustered_spearman")
save_table(correlations, TABLES, "pairwise_clustered_spearman")
ok = correlations.loc[correlations["status"].eq("ok")]
features = sorted(set(ok["feature_left"]) | set(ok["feature_right"]))
matrix = pd.DataFrame(np.eye(len(features)), index=features, columns=features)
for row in ok.itertuples():
    matrix.loc[row.feature_left, row.feature_right] = row.rho
    matrix.loc[row.feature_right, row.feature_left] = row.rho
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(matrix, cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Participant-clustered Spearman structure")
fig.tight_layout()
save_figure(fig, FIGURES, "clustered_spearman_heatmap")
plt.show()

blocked = int(correlations["status"].ne("ok").sum())
goal3_ready = stage_gate(
    "Goal 3",
    True,
    [f"{blocked} pairwise estimands are explicitly under-supported."] if blocked else [],
    "Proceed to Goal 4; use pair-specific denominators in all interpretation.",
)